In [11]:
import numpy as np

NUCLEOTIDES = ["A", "C", "G", "T"]
NUC2IDX     = {n: i for i, n in enumerate(NUCLEOTIDES)}
FASTA_PATH = "436.fasta.fa" 

seq_parts = []
with open(FASTA_PATH) as f:
    for line in f:
        line = line.strip()
        if line.startswith(">"):
            continue
        seq_parts.append(line.upper())
seq = "".join(seq_parts)
dinuc_counts = np.zeros((4, 4), dtype=int)
for k in range(len(seq) - 1):
    a, b = seq[k], seq[k + 1]
    if a in NUC2IDX and b in NUC2IDX:
        dinuc_counts[NUC2IDX[a], NUC2IDX[b]] += 1

print("\nЧастоты динуклеотидов")
print("      " + "  ".join(f"{n:>8}" for n in NUCLEOTIDES))
for i, ni in enumerate(NUCLEOTIDES):
    print(f"  {ni}  " + "  ".join(f"{dinuc_counts[i,j]:>8}" for j in range(4)))
row_sums = dinuc_counts.sum(axis=1, keepdims=True)
P = dinuc_counts / row_sums

print("\nМатрица переходов")
print("         " + "  ".join(f"{n:>9}" for n in NUCLEOTIDES))
for i, ni in enumerate(NUCLEOTIDES):
    print(f"  {ni}  " + "  ".join(f"{P[i,j]:>9.5f}" for j in range(4)))

print("\nПроверка сумм строк")
for i, ni in enumerate(NUCLEOTIDES):
    s  = P[i].sum()
    ok = "✓" if abs(s - 1.0) < 1e-10 else "✗"
    print(f"sum(P[{ni},:]) = {s:.10f}  {ok}")

eigenvalues, eigenvectors = np.linalg.eig(P.T)
idx = np.argmin(np.abs(eigenvalues - 1.0))
pi = eigenvectors[:, idx].real
pi /= pi.sum()

print("\nСтационарное распределение π")
for i, ni in enumerate(NUCLEOTIDES):
    print(f"π({ni}) = {pi[i]:.6f}")
print(f"Σπ= {pi.sum():.10f}  ✓")

pi_check = pi @ P
print("\nПроверка π·P")
for i, ni in enumerate(NUCLEOTIDES):
    d = abs(pi_check[i] - pi[i])
    print(f"(π·P)[{ni}]={pi_check[i]:.6f}  π[{ni}]={pi[i]:.6f}  |Δ|={d:.2e}")

pi_pow = np.full(4, 0.25)
for _ in range(200):
    pi_pow = pi_pow @ P
print(f"\nСтепенной метод: max|Δ| = {np.max(np.abs(pi_pow - pi)):.2e}  ✓")
obs = np.array([seq.count(n) / len(seq) for n in NUCLEOTIDES])

print("\nСтационарное π vs наблюдаемое f")
print(f"  {'':^4}  {'π':>12}  {'f':>12}  {'|Δ|':>10}")
for i, ni in enumerate(NUCLEOTIDES):
    d = abs(pi[i] - obs[i])
    print(f"  {ni:^4}  {pi[i]:>12.6f}  {obs[i]:>12.6f}  {d:>10.6f}")

max_d = np.max(np.abs(pi - obs))
print(f"\n  max|π − f| = {max_d:.6f}")
print(f" {'хорошо согласуется ✓' if max_d < 0.01 else 'заметное расхождение'}")


Частоты динуклеотидов
             A         C         G         T
  A    336495    146252    128806    310686
  C    180417     73547     70298    137492
  G    141089     93585     67169    142862
  T    264237    148370    178432    342540

Матрица переходов
                 A          C          G          T
  A    0.36487    0.15858    0.13967    0.33688
  C    0.39072    0.15928    0.15224    0.29776
  G    0.31726    0.21044    0.15104    0.32125
  T    0.28304    0.15893    0.19113    0.36691

Проверка сумм строк
sum(P[A,:]) = 1.0000000000  ✓
sum(P[C,:]) = 1.0000000000  ✓
sum(P[G,:]) = 1.0000000000  ✓
sum(P[T,:]) = 1.0000000000  ✓

Стационарное распределение π
π(A) = 0.333869
π(C) = 0.167164
π(G) = 0.160992
π(T) = 0.337975
Σπ= 1.0000000000  ✓

Проверка π·P
(π·P)[A]=0.333869  π[A]=0.333869  |Δ|=0.00e+00
(π·P)[C]=0.167164  π[C]=0.167164  |Δ|=8.33e-17
(π·P)[G]=0.160992  π[G]=0.160992  |Δ|=2.78e-17
(π·P)[T]=0.337975  π[T]=0.337975  |Δ|=1.11e-16

Степенной метод: max|Δ| = 1.11e-16 